# Study D, Milestone 5
Separating biological sampling loss, recorder loss and tree-reconstruction error. See `reports/milestone5_decision.md`.

In [1]:
import os, sys, numpy as np, pandas as pd
sys.path.insert(0, '..')
pd.set_option('display.width', 230)
T = '../results/tables/'
have = lambda f: os.path.exists(T + f)

## 1. Where switching information is lost (E17, 480 replicates)
L1 true tree with true branch lengths; L2 same topology, unit edges; L3 reconstructed.

In [2]:
d = pd.read_csv(T+'e17_hierarchy.csv')
print('replicates:', len(d))
display(pd.read_csv(T+'e17_paired_ratios.csv').round(3))

replicates: 480


,level,median_log_ratio,iqr,frac_overestimate,n
0,L2,0.607,0.845,0.790,480
1,L3_greedy,1.009,2.731,0.731,480
2,L3_nj,0.643,1.318,0.771,480
3,L3_upgma,0.708,1.468,0.775,480
4,L3_ens,0.642,1.511,0.665,480


### Star-likeness is already present in the TRUE sampled genealogy

In [3]:
cols = [c for c in d.columns if c.startswith('close_frac')]
display(d.groupby(['recorder','genealogy'])[cols].median().round(4))

close_frac_T1  close_frac_greedy  close_frac_nj  close_frac_upgma
recorder        genealogy                                                                      
cassiopeia_like balanced             0.0162             0.0118         0.0193            0.0136
                bottlenecked         0.0185             0.0122         0.0191            0.0125
gestalt_like    balanced             0.0167             0.0162         0.0320            0.0186
                bottlenecked         0.0164             0.0175         0.0296            0.0189

### Topology recovery, and what recorder capacity buys

In [4]:
rc = [c for c in d.columns if c.startswith('rf_recall')]
display(d.groupby(['recorder','genealogy'])[rc].median().round(3))
display(d.groupby('recorder')[['saturation','missing','n_informative_chars']].median().round(3))

rf_recall_greedy  rf_recall_nj  rf_recall_upgma
recorder        genealogy                                                    
cassiopeia_like balanced                 0.043         0.596            0.596
                bottlenecked             0.032         0.564            0.574
gestalt_like    balanced                 0.064         0.245            0.255
                bottlenecked             0.053         0.191            0.186

,saturation,missing,n_informative_chars
recorder,,,
cassiopeia_like,0.723,0.237,256.0
gestalt_like,0.973,0.196,50.0


### Ensemble versus single trees

In [5]:
for ref in ['k_L3_greedy','k_L3_nj','k_L3_upgma']:
    b = (np.abs(np.log(d.k_L3_ens/d.k_true)) < np.abs(np.log(d[ref]/d.k_true))).mean()
    print(f'ensemble closer to truth than {ref}: {b:.3f}')
es = np.abs(np.log(d[['k_L3_greedy','k_L3_nj','k_L3_upgma']].values/d.k_true.values[:,None])).mean(1)
ee = np.abs(np.log(d.k_L3_ens/d.k_true))
print('mean |log error|: single %.3f, ensemble %.3f' % (es.mean(), ee.mean()))

ensemble closer to truth than k_L3_greedy: 0.702
ensemble closer to truth than k_L3_nj: 0.656
ensemble closer to truth than k_L3_upgma: 0.640
mean |log error|: single 1.663, ensemble 1.368


## 2. Real data: three published reconstructions of the same cells (Quinn 5k)

In [6]:
if have('m5_quinn_variance_by_reconstruction.csv'):
    display(pd.read_csv(T+'m5_quinn_variance_by_reconstruction.csv').round(4))
if have('m5_quinn_reconstruction_geometry.csv'):
    g = pd.read_csv(T+'m5_quinn_reconstruction_geometry.csv')
    display(g.groupby(['mouse','variant'])[['height','mrca_norm','close_frac','L_min','L_max']]
             .median().round(3))

,mouse,variant,n_clones,V_pi_raw,V_logk_raw
0,5k,ensemble,25,0.0094,0.2765
1,5k,hybrid,25,0.0092,0.6339
2,5k,nj,25,0.0083,0.8369
3,5k,nj_priors,25,0.0084,1.8079


height  mrca_norm  close_frac  L_min  L_max
mouse variant                                               
5k    hybrid       15.0      0.196       0.001   10.0   13.0
      nj           28.0      0.220       0.005   15.0   25.0
      nj_priors    35.0      0.244       0.010   21.0   32.0

## 3. Real data: Simeonov, reconstructed from the authors' allele records

In [7]:
if have('m5_simeonov_variance_by_reconstruction.csv'):
    display(pd.read_csv(T+'m5_simeonov_variance_by_reconstruction.csv').round(4))
if have('m5_simeonov_alt_reconstruction_fits.csv'):
    r = pd.read_csv(T+'m5_simeonov_alt_reconstruction_fits.csv')
    display(r.groupby(['mouse','method']).agg(clones=('clone','size'), pi=('pi','median'),
            k=('k','median'), at_bound=('k_at_bound','mean')).round(3))

,mouse,method,n_clones,V_pi_raw
0,1,greedy,10,0.0465
1,1,nj,10,0.0467
2,1,upgma,10,0.0464
3,2,greedy,19,0.0564
4,2,nj,19,0.0564
5,2,upgma,19,0.0563


clones     pi       k  at_bound
mouse method                                 
1     greedy      10  0.335   1.216     0.300
      nj          10  0.336   1.054     0.300
      upgma       10  0.335   1.566     0.400
2     greedy      19  0.438  16.637     0.474
      nj          19  0.437  19.798     0.526
      upgma       19  0.437   1.826     0.368

## 4. The Milestone 4 calibration, completed (300 replicates)

In [8]:
if have('e14_calibration_summary.csv'):
    s = pd.read_csv(T+'e14_calibration_summary.csv')
    print('replicates per scenario:', int(s.n.iloc[0]))
    display(s.pivot(index='scen', columns='test', values='rate').round(3))
    display(pd.read_csv(T+'e14_thresholds.csv').round(2))

replicates per scenario: 300


test,calib_k,calib_pi,chi2_k,chi2_pi,cover_V_pi,cover_pi_profile
scen,,,,,,
H0_k,0.050,0.517,0.013,0.903,0.920,0.918
H0_k_frozen,0.010,0.967,0.010,1.000,0.253,0.965
H0_pi,0.897,0.050,0.833,0.143,0.967,0.935
H1_k,0.883,0.047,0.830,0.163,0.947,0.936
H1_pi,0.077,0.493,0.033,0.867,0.880,0.913
MIS,0.073,0.003,0.040,0.123,0.990,0.719


,scen,thr_boot,thr_chi2
0,H0_pi,32.59,19.68
1,H0_k,17.35,19.68
2,H0_k_frozen,9.81,19.68
